In [1]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import torch
import tensorflow as tf
import math
import re
import collections
from typing import Dict, List, Tuple
import os, pathlib, shutil, random


In [3]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: False


### Load Dataset

In [4]:
# Train - Test - Val Split Placeholder Location
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

In [5]:
# Loading the IMDb dataset for use with Keras
batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(test_dir, batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [6]:
# Padding IMDb reviews to a fixed sequence length
max_length = 600
max_tokens = 30_000
train_ds_no_labels = train_ds.map(lambda x, y: x)
text_vectorization = keras.layers.TextVectorization(max_tokens=max_tokens, split="whitespace", output_mode="int", output_sequence_length=max_length, )
text_vectorization.adapt(train_ds_no_labels)

In [7]:
sequence_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=8)
sequence_val_ds = val_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=8)
sequence_test_ds = test_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=8)

### Model with Embedding Layer

In [8]:
# Building an LSTM sequence model with an Embedding layer
hidden_dim = 64
max_length = 600
max_tokens = 30_000
inputs = keras.Input(shape=(max_length,), dtype="int32")
x = keras.layers.Embedding(input_dim=max_tokens, output_dim=hidden_dim, mask_zero=True, )(inputs)
x = keras.layers.Bidirectional(keras.layers.LSTM(hidden_dim))(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_embedding")
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"], )
model.summary()

Model: "lstm_with_embedding"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 600)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 600, 64)   │  1,920,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 600)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     66,048 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │        129 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,986,177 (7.58 MB)

 Trainable params: 1,986,177 (7.58 MB)

 Non-trainable params: 0 (0.00 B)

### Training

In [9]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)
model.fit(sequence_train_ds, validation_data=sequence_val_ds, epochs=10, callbacks=[early_stopping],)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 227s 360ms/step - accuracy: 0.7531 - loss: 0.4952 - val_accuracy: 0.8668 - val_loss: 0.3300
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 234s 374ms/step - accuracy: 0.9115 - loss: 0.2279 - val_accuracy: 0.8824 - val_loss: 0.3206
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 235s 376ms/step - accuracy: 0.9546 - loss: 0.1251 - val_accuracy: 0.8818 - val_loss: 0.3830
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 273s 436ms/step - accuracy: 0.9789 - loss: 0.0681 - val_accuracy: 0.8694 - val_loss: 0.4971


### Evaluation

In [10]:
test_loss, test_acc = model.evaluate(sequence_test_ds)
test_loss, test_acc

782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.8705 - loss: 0.3470


(0.3470210134983063, 0.8705199956893921)